# Preparation of the cell-cell communication tables for Fig. 4

Not needed to make the figures. Documents how the cell-cell communication tables in `data/` (Supplementary Tables 10-17, `*_ccc_*.csv`) were extracted from the results of the
cell-cell communication analysis (`full/ccc_progression/`, notebooks `15a_ccc_progression_compute.ipynb` and
`15b_ccc_progression_figures.ipynb`), which are not distributed here. Only the rows and columns drawn in Fig. 4 are kept.

In [1]:
import json
from pathlib import Path

import pandas as pd

CCC_TABLES = Path("../full/ccc_progression/tables")
DATA = Path("../data")

SIM = "Macrophages_Smoking-induced"
GROUP_KEY = "Emphysema Progression Type"

summary = json.loads((CCC_TABLES / "run_summary.json").read_text())
progression_factor = summary["progression_factor"]
focus_receptors = summary["focus_receptors"]
exemplar_interactions = summary["exemplar_features"]
print(progression_factor, focus_receptors, exemplar_interactions)

Factor1 ['TLR2', 'NRP1'] {'TLR2': 'DC1&Macrophages_Smoking-induced:HMGB1^TLR2', 'NRP1': 'DC2&Macrophages_Smoking-induced:VEGFA^NRP1'}


## Samples: group and progression factor score (Fig. 4b, e-f and supplementary)

In [2]:
samples = pd.read_csv(CCC_TABLES / "samples.csv", index_col="sample_id")
samples[[GROUP_KEY, "progression_factor"]].to_csv(DATA / "11_ccc_samples.csv")

## MOFAtalk factors: association with progression and variance explained per view (Fig. 4a, c)

In [3]:
factor_tests = pd.read_csv(CCC_TABLES / "mofa_factor_progression_tests.csv", index_col=0)
factor_tests[["t", "pvalue"]].to_csv(DATA / "10_ccc_mofa_factor_progression_tests.csv")

r2_per_view = pd.read_csv(CCC_TABLES / "mofa_r2_per_view.csv")
r2_per_view = r2_per_view[r2_per_view["Factor"] == progression_factor]
r2_per_view[["source", "target", "R2"]].to_csv(DATA / "12_ccc_mofa_progression_factor_r2_per_view.csv", index=False)

## Receptor-centred summary of the SIM interactions (Fig. 4d) and the tests of the exemplar interactions (Fig. 4e-f)

In [4]:
sim_receptor_summary = pd.read_csv(CCC_TABLES / "sim_receptor_summary.csv", index_col=0)
sim_receptor_summary[["n_nominal", "mean_score_t", "sim_expression_t"]].to_csv(DATA / "13_ccc_sim_receptor_summary.csv")

In [5]:
interaction_tests = pd.read_csv(CCC_TABLES / "interaction_score_progression_tests.csv", index_col=0)
interaction_tests.loc[list(exemplar_interactions.values()), ["source", "target", "ligand_complex", "receptor_complex", "t", "pvalue"]].to_csv(DATA / "14_ccc_exemplar_interaction_tests.csv")

## Per-sample scores of the exemplar interactions (Fig. 4e-f) and expression of the focus receptors in SIM pseudobulks (supplementary)

In [6]:
interaction_scores = pd.read_parquet(CCC_TABLES / "interaction_scores.parquet")
interaction_scores[list(exemplar_interactions.values())].to_csv(DATA / "15_ccc_exemplar_interaction_scores.csv")

log_cpm = pd.read_csv(CCC_TABLES / "macrophage_pseudobulk_log_cpm.csv")
log_cpm = log_cpm[log_cpm["cell_type_reannotated"] == SIM].set_index("sample_id")
log_cpm[focus_receptors].to_csv(DATA / "16_ccc_sim_pseudobulk_log_cpm.csv")

edger = pd.read_csv(CCC_TABLES / "pseudobulk_edger_lr_genes.csv", index_col=0)
edger = edger[(edger["cell_type_reannotated"] == SIM) & edger.index.isin(focus_receptors)]
edger[["logfc", "pvalue"]].rename_axis("gene").to_csv(DATA / "17_ccc_sim_edger_focus_receptors.csv")

In [7]:
for file in sorted(DATA.glob("*_ccc_*.csv"), key=lambda p: int(p.name.split("_")[0])):
    print(f"{file.name:50} {file.stat().st_size / 1024:7.0f} kB")

10_ccc_mofa_factor_progression_tests.csv                 0 kB
11_ccc_samples.csv                                       3 kB
12_ccc_mofa_progression_factor_r2_per_view.csv          13 kB
13_ccc_sim_receptor_summary.csv                          6 kB
14_ccc_exemplar_interaction_tests.csv                    0 kB
15_ccc_exemplar_interaction_scores.csv                   2 kB
16_ccc_sim_pseudobulk_log_cpm.csv                        3 kB
17_ccc_sim_edger_focus_receptors.csv                     0 kB
